# 023 — Training of `unet` variants

Trains `unet_v2`, `unet_restormer`, `unet_dilated`, and `unet_v2_dilated`.  
Four independent architectural variants of `unet`.

- `unet_v2`  
the standard UNet with three independently-toggleable modifications, all off by default and matching `unet.py` exactly when off: 
    - `use_strided_conv` (learned stride-2 convolution instead of `MaxPool2D`)
    - `use_upsample_conv` (bilinear upsample + `Conv2D` instead of `Conv2DTranspose`, avoids checkerboard artifacts)
    - `dropout_rate` (`SpatialDropout2D` at the bottleneck and first decoder block only) 
- `unet_restormer`   
the standard UNet with a Restormer transformer block inserted at the bottleneck, giving the network a global receptive field via channel-wise self-attention (linear cost in image size, not quadratic)  

- `unet_dilated`/`unet_v2_dilated`  
replace the plain bottleneck with a multi-scale ASPP-style block that widens the receptive field without adding downsampling.  
`unet_v2_dilated` is trained with the same `use_strided_conv`/`use_upsample_conv`/`dropout_rate`

Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset split
**Split used project-wide:**  
- Real **artworks** are grouped and kept entirely within one fold, exactly as a plain grouped split would do, so no painting leaks across train/val/test.  
- The **mockup** groups (listed in `settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from. They are split at the individual-pair level, with only `settings.MOCKUP_TEST_RATIO` (default 5%) held out for test. 


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Train all variants

Each architecture trains in its own subprocess.  
Checkpoints and hystories go to `models/deterministic/<arch>/`.

In [ ]:
import json
import subprocess

DET_VARIANTS = {
    "unet_v2": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
    "unet_restormer": dict(num_heads=8, ffn_expansion_factor=2),
    "unet_dilated": dict(),
    "unet_v2_dilated": dict(
        use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2
    ),
}
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "deterministic"
LOG_DIR = settings.LOGS_DIR / "deterministic"

histories: dict = {}

for arch, kwargs in DET_VARIANTS.items():
    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        arch,
        "--epochs",
        str(EPOCHS),
        "--model-dir",
        str(MODEL_DIR),
        "--log-dir",
        str(LOG_DIR),
        "--kwargs",
        json.dumps(kwargs),
    ]
    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = MODEL_DIR / arch / "history.json"
    histories[arch] = json.loads(history_path.read_text())

    best_val_loss = min(histories[arch]["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 3. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 4. Summary


In [ ]:
for arch in DET_VARIANTS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")